### Description of the Notebook - Governance Officer

In [2]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

### Loading the clean datasets

In [4]:
app_df = pd.read_csv('applications_clean.csv')
spend_df = pd.read_csv('spending_behavior.csv')

In [5]:
# Display the first 10 rows of the applications dataset
print("Applications Clean Dataset:")
display(app_df.head(10))

# Display the first 10 rows of the spending behavior dataset
print("\nSpending Behavior Dataset:")
display(spend_df.head(10))

Applications Clean Dataset:


,app_id,processing_timestamp,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,...,financials_savings_balance,decision_loan_approved,decision_rejection_reason,loan_purpose,decision_interest_rate,decision_approved_amount,notes,data_quality_flag,missing_fields,flag_note
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,...,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,...,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,...,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,NaN,NaN
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,...,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,NaN,NaN
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,...,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019.0,110000.0,...,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,1990-01-28,10022.0,55000.0,...,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,NaN,NaN
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,Female,1991-10-11,90223.0,82000.0,...,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,NaN,NaN
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044.0,69000.0,...,15974,False,algorithm_risk_score,NaN,NaN,NaN,RESUBMISSION,NaN,NaN,NaN
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080.0,55000.0,...,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Spending Behavior Dataset:


,app_id,category,amount
0,app_200,Shopping,480
1,app_200,Rent,790
2,app_200,Alcohol,247
3,app_037,Rent,608
4,app_037,Dining,96
5,app_037,Healthcare,243
6,app_215,Rent,109
7,app_024,Fitness,575
8,app_184,Entertainment,463
9,app_275,Entertainment,571


# PII IDENTIFICATION 

The given dataset contains different types of PII. The PIIs can be distingueshed in: 

Direct Identifiers: PII that can be obviously and directly linked to a subject
- Name
- Email
- Social Security Number
- Ip address

Quasi identifiers: PIIs that in combination allow potenial linkage to a data subject
- Gender
- Date of Birth
- ZIP Code
- Annual Income

Several violations by NovaCred in the management and use of personal data must be reported. 

**Elimination of IP Address variable:** according to Article 5(1)(c) of the GDPR, controllers can only collect and process personal data that are adequate, relevant, and limited to what is necessary for the purposes for which they are processed: in this case, the IP address feature  violates the data minimisation principle, as it clearly trespasses the purpose of a finanzial company. Therefore, it will be removed from the dataset.

In [8]:
# --- GOVERNANCE OFFICER TASK: DATA MINIMIZATION ---
# Removing the IP address as it violates GDPR Art. 5(1)(c)

if 'applicant_info_ip_address' in app_df.columns:
    app_df = app_df.drop(columns=['applicant_info_ip_address'])
    print("SUCCESS: 'applicant_info_ip_address' has been removed.")
else:
    print("NOTE: Column not found or already removed.")

# Verify the current columns
print("\nCurrent columns in app_df:")
print(app_df.columns.tolist())

SUCCESS: 'applicant_info_ip_address' has been removed.

Current columns in app_df:
['app_id', 'processing_timestamp', 'applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_gender', 'applicant_info_date_of_birth', 'applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'decision_loan_approved', 'decision_rejection_reason', 'loan_purpose', 'decision_interest_rate', 'decision_approved_amount', 'notes', 'data_quality_flag', 'missing_fields', 'flag_note']
